# M3: Entrenamiento de Keypoints de Ruedas / Ejes (YOLO11s-pose)
Un keypoint `centro` por cada rueda visible desde el lateral.


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/evidencias /content/drive/MyDrive/peaje/modelos /content/drive/MyDrive/peaje/runs
!git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje 2>/dev/null || git -C /content/Control_Peaje pull
%cd /content/Control_Peaje
!pip -q install -r entrenamiento/requirements.txt


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = 'tu-workspace'
PROYECTO = 'peaje-ruedas-pose'
VERSION = 1

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11')


In [ ]:
import yaml
yaml_path = f'{ds.location}/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['kpt_shape'] = [1, 3]
cfg['flip_idx'] = [0]
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('data.yaml verificado:', cfg)


In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s-pose.pt')
modelo.train(
    data=yaml_path,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    project='/content/drive/MyDrive/peaje/runs',
    name='m3_pose',
)


In [ ]:
import glob, json, os, shutil
import numpy as np
best_pt = '/content/drive/MyDrive/peaje/runs/m3_pose/weights/best.pt'
mejor = YOLO(best_pt)
metricas = mejor.val(data=yaml_path)

# Error en el conteo de ejes sobre el split de validacion
errores = []
exactos = 0
imgs_val = sorted(glob.glob(f'{ds.location}/valid/images/*'))
for img_p in imgs_val:
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = f'{ds.location}/valid/labels/{stem}.txt'
    gt = sum(1 for l in open(lbl_p) if l.strip()) if os.path.exists(lbl_p) else 0
    pred_r = mejor(img_p, verbose=False, conf=0.5)[0]
    pred_cnt = len(pred_r.boxes) if pred_r.boxes is not None else 0
    errores.append(abs(gt - pred_cnt))
    if gt == pred_cnt:
        exactos += 1

resumen = {
    'pose_map50': float(metricas.pose.map50),
    'box_map50': float(metricas.box.map50),
    'imagenes_val': len(imgs_val),
    'mae_ejes': float(np.mean(errores)) if errores else None,
    'exactitud_conteo_ejes': (exactos / len(imgs_val)) if imgs_val else None,
}
print(json.dumps(resumen, indent=2))
with open('/content/drive/MyDrive/peaje/evidencias/metricas_m3.json', 'w') as f:
    json.dump(resumen, f, indent=2)
shutil.copy(best_pt, '/content/drive/MyDrive/peaje/modelos/m3_pose.pt')
print('Modelo guardado en Drive: m3_pose.pt')
